# Learning Algorithms

Notebook này trình bày quy trình thực hành trực tiếp cho nhóm **Learning Algorithms** trên Grid-world 8x8.

Learning Algorithms không biết model đầy đủ $P,r$. Agent học từ sampled transitions:

$$(S_t,A_t,R_{t+1},S_{t+1})$$

Trong training, các thuật toán learning chỉ dùng `reset()` và `step(action)`, không dùng `get_transitions()`.

Các thuật toán:

1. TD(0)
2. TD(lambda)
3. SARSA
4. Q-learning

TD(0), TD(lambda) là prediction methods. SARSA và Q-learning là control methods. SARSA là on-policy; Q-learning là off-policy.

## Mục lục

- Setup imports
- Thiết kế Grid-world 8x8
- Chuẩn bị baseline PE/VI
- TD(0)
- TD(lambda)
- SARSA
- Q-learning
- Đánh giá và so sánh Learning models
- Thực nghiệm mở rộng tùy chọn
- Tổng kết Learning

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from envs.learning_grid_world import LearningGridWorld
from envs.planning_grid_world import PlanningGridWorld
from agents.planning import PolicyEvaluation, ValueIteration
from agents.learning import TDZero, TDLambda, SARSA, QLearning
from utils.metrics import mean_squared_error, policy_agreement
from utils.visualization import (
    plot_value_heatmap,
    plot_policy_arrows,
    plot_learning_curve,
    plot_moving_average,
    plot_episode_steps,
    plot_td_error_curve,
    plot_td_mse_curve,
    plot_success_trap_curves,
    plot_comparison_bar,
)

NOTEBOOK_FIGURE_DIR = PROJECT_ROOT / "report" / "figures" / "notebook_demos" / "learning"
NOTEBOOK_VERBOSE = 2
NOTEBOOK_LOG_INTERVAL = 100
NOTEBOOK_WINDOW_SIZE = 100
LEARNING_EPISODES = 1000
MAX_STEPS_PER_EPISODE = 256
RANDOM_SEED = 42

In [ ]:
def show_metrics(metrics, keys=None):
    if metrics is None:
        print("No metrics available.")
        return
    selected = metrics if keys is None else {key: metrics.get(key) for key in keys}
    for key, value in selected.items():
        print(f"{key}: {value}")


def load_json(path):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def show_png(path, width=720):
    path = Path(path)
    if not path.exists():
        print(f"Missing figure: {path}")
        return
    display(Image(filename=str(path), width=width))

## Thiết kế Grid-world 8x8

Grid-world là môi trường rời rạc. State là tọa độ ô `(row, col)`. Action là các hành động di chuyển như `up`, `down`, `left`, `right`.

- **Goal** là trạng thái kết thúc tốt.
- **Trap** là trạng thái kết thúc xấu.
- **Wall** là ô không thể đi qua.
- **Reward** gồm step reward, goal reward, trap reward và wall/boundary penalty theo config hiện tại.
- **Terminal states** gồm Goal và Trap.

Grid-world có thể được mô hình hóa như một MDP:

$$M=(S,A,P,r,\gamma)$$

Planning dùng model đầy đủ qua `get_transitions(state, action)`. Learning chỉ quan sát sample qua `reset()` và `step(action)` trong training.

In [ ]:
learning_env = LearningGridWorld(seed=RANDOM_SEED)
learning_env

In [ ]:
print("Grid size:", learning_env.grid_size)
print("Start state:", learning_env.start_state)
print("Goal states:", learning_env.goal_states)
print("Trap states:", learning_env.trap_states)
print("Wall states:", learning_env.wall_states)
print("Actions:", learning_env.actions)
print("Reward config:", learning_env.reward_config)
print("Gamma:", learning_env.gamma)
print("Number of states:", learning_env.num_states())

In [ ]:
# TODO: Nếu đã có helper visualize layout riêng, gọi tại đây.
# Ví dụ: plot_grid_world_layout(learning_env, title="Grid-world 8x8")
# Hiện project có thể visualize value/policy sau khi train model.

## Chuẩn bị baseline cho đánh giá Learning

TD methods cần baseline $V^\pi$ từ PolicyEvaluation. Control methods cần baseline $V^*$ và $\pi^*$ từ ValueIteration.

Các baseline này chỉ dùng để **đánh giá** model sau training. Chúng không được dùng trong quá trình training model-free của TD/SARSA/Q-learning.

In [ ]:
baseline_env = PlanningGridWorld(seed=RANDOM_SEED)

baseline_policy_eval = PolicyEvaluation(
    env=baseline_env,
    policy=None,  # uniform random policy mặc định, cùng tinh thần với TD policy mặc định
    theta=1e-6,
    max_iterations=1000,
    verbose=0,
)
baseline_policy_eval.run()
baseline_v_pi = baseline_policy_eval.get_value_function()

baseline_vi = ValueIteration(
    env=baseline_env,
    theta=1e-6,
    max_iterations=1000,
    verbose=0,
)
baseline_vi.run()
baseline_v_star = baseline_vi.get_value_function()
baseline_pi_star = baseline_vi.get_policy()

print("Policy Evaluation baseline status:", baseline_policy_eval.get_metrics().get("status"))
print("Value Iteration baseline status:", baseline_vi.get_metrics().get("status"))

## TD(0)

**Mục tiêu:** học $V^\pi$ từ sample.

TD error:

$$\delta_t=R_{t+1}+\gamma V(S_{t+1})-V(S_t)$$

Update:

$$V(S_t)\leftarrow V(S_t)+\alpha\delta_t$$

TD(0) không dùng `get_transitions()` khi train. Baseline đánh giá là PolicyEvaluation.

In [ ]:
td_zero_env = LearningGridWorld(seed=RANDOM_SEED)

td_zero = TDZero(
    env=td_zero_env,
    episodes=LEARNING_EPISODES,
    alpha=0.1,
    gamma=td_zero_env.gamma,
    policy=None,
    baseline_value_function=baseline_v_pi,
    verbose=NOTEBOOK_VERBOSE,
    log_interval=NOTEBOOK_LOG_INTERVAL,
    seed=RANDOM_SEED,
    max_steps_per_episode=MAX_STEPS_PER_EPISODE,
)

td_zero_result = td_zero.train()
td_zero_values = td_zero.get_value_function()
td_zero_metrics = td_zero.get_metrics()

In [ ]:
# Đánh giá trực tiếp từ model vừa train.
td_zero_mse_direct = mean_squared_error(td_zero_values, baseline_v_pi)
td_zero_metrics["notebook_mse_vs_policy_evaluation"] = td_zero_mse_direct

show_metrics(td_zero_metrics, keys=[
    "episodes",
    "environment_steps",
    "final_mean_absolute_td_error",
    "mse_vs_policy_evaluation",
    "notebook_mse_vs_policy_evaluation",
    "runtime_sec",
])

In [ ]:
td0_mse_path = NOTEBOOK_FIGURE_DIR / "td_zero_mse_vs_pe.png"
td0_error_path = NOTEBOOK_FIGURE_DIR / "td_zero_td_error.png"
td0_value_path = NOTEBOOK_FIGURE_DIR / "td_zero_value_heatmap.png"

plot_td_mse_curve(
    td_zero_metrics["mse_checkpoint_episodes"],
    td_zero_metrics["mse_vs_policy_evaluation_checkpoints"],
    "TD(0) - MSE vs Policy Evaluation",
    td0_mse_path,
)
plot_td_error_curve(td_zero_metrics["mean_absolute_td_error_per_episode"], "TD(0) TD Error", td0_error_path)
plot_value_heatmap(td_zero_values, td_zero_env, "TD(0) - V^pi Estimate", td0_value_path)

show_png(td0_mse_path)
show_png(td0_error_path)
show_png(td0_value_path)

**Nhận xét cần điền sau khi chạy:**

- TD error dao động như thế nào?
- MSE vs PolicyEvaluation có giảm không?
- Value function học được có hợp lý theo vị trí Goal/Trap không?

## TD(lambda)

**Mục tiêu:** mở rộng TD(0) bằng eligibility traces.

Eligibility trace:

$$e_t(s)=\gamma\lambda e_{t-1}(s)+\mathbf{1}\{S_t=s\}$$

Update:

$$V(s)\leftarrow V(s)+\alpha\delta_t e_t(s)$$

$\lambda=0$ gần TD(0). $\lambda$ càng lớn thì trace càng dài. TD(lambda) không đảm bảo luôn tốt hơn TD(0); cần tuning $\lambda$ và $\alpha$.

In [ ]:
td_lambda_env = LearningGridWorld(seed=RANDOM_SEED)

td_lambda = TDLambda(
    env=td_lambda_env,
    episodes=LEARNING_EPISODES,
    alpha=0.1,
    gamma=td_lambda_env.gamma,
    lambda_=0.7,
    policy=None,
    baseline_value_function=baseline_v_pi,
    verbose=NOTEBOOK_VERBOSE,
    log_interval=NOTEBOOK_LOG_INTERVAL,
    seed=RANDOM_SEED,
    max_steps_per_episode=MAX_STEPS_PER_EPISODE,
)

td_lambda_result = td_lambda.train()
td_lambda_values = td_lambda.get_value_function()
td_lambda_metrics = td_lambda.get_metrics()

In [ ]:
td_lambda_mse_direct = mean_squared_error(td_lambda_values, baseline_v_pi)
td_lambda_metrics["notebook_mse_vs_policy_evaluation"] = td_lambda_mse_direct

show_metrics(td_lambda_metrics, keys=[
    "episodes",
    "lambda",
    "environment_steps",
    "final_mean_absolute_td_error",
    "mse_vs_policy_evaluation",
    "notebook_mse_vs_policy_evaluation",
    "runtime_sec",
])

In [ ]:
tdl_mse_path = NOTEBOOK_FIGURE_DIR / "td_lambda_mse_vs_pe.png"
tdl_error_path = NOTEBOOK_FIGURE_DIR / "td_lambda_td_error.png"
tdl_value_path = NOTEBOOK_FIGURE_DIR / "td_lambda_value_heatmap.png"

plot_td_mse_curve(
    td_lambda_metrics["mse_checkpoint_episodes"],
    td_lambda_metrics["mse_vs_policy_evaluation_checkpoints"],
    "TD(lambda) - MSE vs Policy Evaluation",
    tdl_mse_path,
)
plot_td_error_curve(td_lambda_metrics["mean_absolute_td_error_per_episode"], "TD(lambda) TD Error", tdl_error_path)
plot_value_heatmap(td_lambda_values, td_lambda_env, "TD(lambda) - V^pi Estimate", tdl_value_path)

show_png(tdl_mse_path)
show_png(tdl_error_path)
show_png(tdl_value_path)

**Nhận xét cần điền sau khi chạy:**

- TD(lambda) khác TD(0) ở MSE curve như thế nào?
- Với `lambda_=0.7`, thuật toán có ổn định không?
- Cần thử thêm các lambda nào trong phần sensitivity?

## SARSA

**Mục tiêu:** on-policy control.

$$Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\alpha[R_{t+1}+\gamma Q(S_{t+1},A_{t+1})-Q(S_t,A_t)]$$

SARSA chọn action bằng epsilon-greedy, chọn next action cũng bằng policy hiện tại, rồi update theo actual next action $A_{t+1}$.

Baseline đánh giá là ValueIteration.

In [ ]:
sarsa_env = LearningGridWorld(seed=RANDOM_SEED)

sarsa = SARSA(
    env=sarsa_env,
    episodes=LEARNING_EPISODES,
    alpha=0.1,
    gamma=sarsa_env.gamma,
    epsilon=0.1,
    verbose=NOTEBOOK_VERBOSE,
    log_interval=NOTEBOOK_LOG_INTERVAL,
    window_size=NOTEBOOK_WINDOW_SIZE,
    seed=RANDOM_SEED,
    max_steps_per_episode=MAX_STEPS_PER_EPISODE,
)

sarsa_result = sarsa.train()
sarsa_q = sarsa.get_q_table()
sarsa_values = sarsa.get_value_function()
sarsa_policy = sarsa.get_policy()
sarsa_metrics = sarsa.get_metrics()

In [ ]:
non_terminal_states = [state for state in baseline_env.get_states() if not baseline_env.is_terminal(state)]

sarsa_mse_vs_vi = mean_squared_error(sarsa_values, baseline_v_star)
sarsa_agreement_vs_vi = policy_agreement(sarsa_policy, baseline_pi_star, non_terminal_states)
sarsa_metrics["notebook_mse_vs_value_iteration"] = sarsa_mse_vs_vi
sarsa_metrics["notebook_policy_agreement_vs_value_iteration"] = sarsa_agreement_vs_vi

show_metrics(sarsa_metrics, keys=[
    "episodes",
    "environment_steps",
    "training_avg_return",
    "final_window_avg_return",
    "final_window_success_rate",
    "final_window_trap_rate",
    "notebook_mse_vs_value_iteration",
    "notebook_policy_agreement_vs_value_iteration",
    "runtime_sec",
])

In [ ]:
sarsa_return_path = NOTEBOOK_FIGURE_DIR / "sarsa_episode_returns.png"
sarsa_success_path = NOTEBOOK_FIGURE_DIR / "sarsa_success_trap.png"
sarsa_policy_path = NOTEBOOK_FIGURE_DIR / "sarsa_policy.png"
sarsa_value_path = NOTEBOOK_FIGURE_DIR / "sarsa_value_heatmap.png"

plot_learning_curve(sarsa_metrics["episode_returns"], "SARSA Episode Returns", sarsa_return_path)
plot_success_trap_curves(
    sarsa_metrics["window_metric_episodes"],
    sarsa_metrics["window_success_rates"],
    sarsa_metrics["window_trap_rates"],
    "SARSA Success/Trap Rates",
    sarsa_success_path,
)
plot_policy_arrows(sarsa_policy, sarsa_env, "SARSA Learned Policy", sarsa_policy_path)
plot_value_heatmap(sarsa_values, sarsa_env, "SARSA max_a Q(s,a)", sarsa_value_path)

show_png(sarsa_return_path)
show_png(sarsa_success_path)
show_png(sarsa_policy_path)
show_png(sarsa_value_path)

**Nhận xét cần điền sau khi chạy:**

- SARSA cải thiện return qua episode như thế nào?
- Success/trap rate thay đổi ra sao?
- On-policy update ảnh hưởng policy học được như thế nào?

## Q-learning

**Mục tiêu:** off-policy optimal control.

$$Q(S_t,A_t)\leftarrow Q(S_t,A_t)+\alpha[R_{t+1}+\gamma\max_{a'}Q(S_{t+1},a')-Q(S_t,A_t)]$$

Behavior policy có thể epsilon-greedy, nhưng target dùng greedy action ở next state. Thuật toán học hướng về $Q^*$.

Baseline đánh giá là ValueIteration.

In [ ]:
q_env = LearningGridWorld(seed=RANDOM_SEED)

q_learning = QLearning(
    env=q_env,
    episodes=LEARNING_EPISODES,
    alpha=0.1,
    gamma=q_env.gamma,
    epsilon=0.1,
    verbose=NOTEBOOK_VERBOSE,
    log_interval=NOTEBOOK_LOG_INTERVAL,
    window_size=NOTEBOOK_WINDOW_SIZE,
    seed=RANDOM_SEED,
    max_steps_per_episode=MAX_STEPS_PER_EPISODE,
)

q_result = q_learning.train()
q_table = q_learning.get_q_table()
q_values = q_learning.get_value_function()
q_policy = q_learning.get_policy()
q_metrics = q_learning.get_metrics()

In [ ]:
q_mse_vs_vi = mean_squared_error(q_values, baseline_v_star)
q_agreement_vs_vi = policy_agreement(q_policy, baseline_pi_star, non_terminal_states)
q_metrics["notebook_mse_vs_value_iteration"] = q_mse_vs_vi
q_metrics["notebook_policy_agreement_vs_value_iteration"] = q_agreement_vs_vi

show_metrics(q_metrics, keys=[
    "episodes",
    "environment_steps",
    "training_avg_return",
    "final_window_avg_return",
    "final_window_success_rate",
    "final_window_trap_rate",
    "notebook_mse_vs_value_iteration",
    "notebook_policy_agreement_vs_value_iteration",
    "runtime_sec",
])

In [ ]:
q_return_path = NOTEBOOK_FIGURE_DIR / "q_learning_episode_returns.png"
q_success_path = NOTEBOOK_FIGURE_DIR / "q_learning_success_trap.png"
q_policy_path = NOTEBOOK_FIGURE_DIR / "q_learning_policy.png"
q_value_path = NOTEBOOK_FIGURE_DIR / "q_learning_value_heatmap.png"

plot_learning_curve(q_metrics["episode_returns"], "Q-learning Episode Returns", q_return_path)
plot_success_trap_curves(
    q_metrics["window_metric_episodes"],
    q_metrics["window_success_rates"],
    q_metrics["window_trap_rates"],
    "Q-learning Success/Trap Rates",
    q_success_path,
)
plot_policy_arrows(q_policy, q_env, "Q-learning Learned Policy", q_policy_path)
plot_value_heatmap(q_values, q_env, "Q-learning max_a Q(s,a)", q_value_path)

show_png(q_return_path)
show_png(q_success_path)
show_png(q_policy_path)
show_png(q_value_path)

**Nhận xét cần điền sau khi chạy:**

- Q-learning cải thiện return qua episode như thế nào?
- Greedy target khác SARSA ra sao?
- Policy agreement với ValueIteration có cần bằng 1.0 để policy hữu ích không?

## Đánh giá và so sánh Learning models

Cell dưới đây đánh giá trực tiếp từ các model vừa train. Logs JSON chỉ là optional/reference, không phải luồng chính.

In [ ]:
learning_comparison = {
    "TDZero_mse_vs_PE": td_zero_metrics.get("notebook_mse_vs_policy_evaluation"),
    "TDLambda_mse_vs_PE": td_lambda_metrics.get("notebook_mse_vs_policy_evaluation"),
    "SARSA_window_return": sarsa_metrics.get("final_window_avg_return"),
    "QLearning_window_return": q_metrics.get("final_window_avg_return"),
    "SARSA_window_success": sarsa_metrics.get("final_window_success_rate"),
    "QLearning_window_success": q_metrics.get("final_window_success_rate"),
    "SARSA_mse_vs_VI": sarsa_metrics.get("notebook_mse_vs_value_iteration"),
    "QLearning_mse_vs_VI": q_metrics.get("notebook_mse_vs_value_iteration"),
    "SARSA_policy_agreement": sarsa_metrics.get("notebook_policy_agreement_vs_value_iteration"),
    "QLearning_policy_agreement": q_metrics.get("notebook_policy_agreement_vs_value_iteration"),
}
show_metrics(learning_comparison)

In [ ]:
control_return_path = NOTEBOOK_FIGURE_DIR / "control_window_return_comparison.png"
control_success_path = NOTEBOOK_FIGURE_DIR / "control_window_success_comparison.png"
control_agreement_path = NOTEBOOK_FIGURE_DIR / "control_policy_agreement_comparison.png"

plot_comparison_bar(
    {
        "SARSA": learning_comparison["SARSA_window_return"],
        "Q-learning": learning_comparison["QLearning_window_return"],
    },
    "Control Final Window Return",
    control_return_path,
    ylabel="Return",
)
plot_comparison_bar(
    {
        "SARSA": learning_comparison["SARSA_window_success"],
        "Q-learning": learning_comparison["QLearning_window_success"],
    },
    "Control Final Window Success Rate",
    control_success_path,
    ylabel="Success rate",
)
plot_comparison_bar(
    {
        "SARSA": learning_comparison["SARSA_policy_agreement"],
        "Q-learning": learning_comparison["QLearning_policy_agreement"],
    },
    "Control Policy Agreement vs Value Iteration",
    control_agreement_path,
    ylabel="Agreement",
)

show_png(control_return_path)
show_png(control_success_path)
show_png(control_agreement_path)

## Thực nghiệm mở rộng tùy chọn

Các phân tích sau không chạy mặc định trong notebook để tránh thời gian chạy dài:

- TD(lambda) sweep: thử `lambda_values = [0.0, 0.3, 0.5, 0.7, 0.9]`.
- Epsilon sensitivity: thử nhiều `epsilon` cho SARSA/Q-learning.
- Multi-seed smoke: chạy nhiều seed để kiểm tra ổn định nhẹ.

Có thể chạy qua CLI:

```bash
python3 run_experiments.py --verbose 1 --run-td-lambda-sweep
python3 run_experiments.py --verbose 1 --run-control-sensitivity
python3 run_experiments.py --verbose 1 --run-multiseed-smoke
```

TODO: nếu cần phân tích trong notebook, thêm cell loop nhỏ và tái sử dụng các class đã import, không reimplement thuật toán.

In [ ]:
# Optional/reference only: đọc logs nếu đã chạy CLI sensitivity.
learning_log_dir = PROJECT_ROOT / "logs" / "learning"
td_lambda_sweep = load_json(learning_log_dir / "td_lambda_sweep.json")
control_sensitivity = load_json(learning_log_dir / "control_sensitivity.json")
multiseed_smoke = load_json(learning_log_dir / "multiseed_smoke.json")

if td_lambda_sweep:
    print("TD(lambda) best lambda:", td_lambda_sweep.get("best_lambda_by_mse"))
if control_sensitivity:
    print("SARSA best epsilon:", control_sensitivity.get("sarsa_best_epsilon_by_window_return"))
    print("Q-learning best epsilon:", control_sensitivity.get("q_learning_best_epsilon_by_window_return"))
if multiseed_smoke:
    print("Multi-seed aggregate:", multiseed_smoke.get("aggregate"))

## Tổng kết Learning

TD methods học value từ sample. TD(lambda) dùng eligibility traces để lan truyền TD error về các state trước đó. SARSA học on-policy, còn Q-learning học off-policy.

PolicyEvaluation và ValueIteration đóng vai trò baseline để đánh giá kết quả, không dùng để training model-free algorithms. Kết quả learning phụ thuộc số episode, hyperparameters, exploration và random seed.

TODO: Viết phần phân tích cuối cùng sau khi chạy toàn bộ notebook, chọn figures phù hợp và so sánh với Planning notebook.